# 10 — Ergebnisse darstellen

Dieses Notebook liest ausschließlich die sieben Ergebnisdateien eines abgeschlossenen
Laufs. Es schätzt keine Modelle und greift nicht auf Firmen- oder Rasterzeilen zu.
H4 umfasst Linien, Abfahrtsfrequenz und den Indikator ohne Haltestelle. H5 umfasst
Sparteneffekte und deren gemeinsamen Vergleich. Unsicherheiten und Modellgrenzen
bleiben in den Tabellen und im Überblick sichtbar.


In [ ]:
from pathlib import Path
import json
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

PROJECT_DIR = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
                   if (p / "ANAL").is_dir() and (p / "FIG").is_dir())
CELL_SIZE = 100
RUN_NAME = "final"
RESULT_DIR = PROJECT_DIR / "RESULTS" / f"{CELL_SIZE}m" / RUN_NAME
OUTPUT_DIR = PROJECT_DIR / "FIG" / "outputs" / f"{CELL_SIZE}m" / RUN_NAME
sys.path.insert(0, str(PROJECT_DIR))
from FIG.plotting.apa7 import set_apa7_style, apa7_axes
set_apa7_style()


## 1. Ergebnisdateien prüfen und einlesen


In [ ]:
info = json.loads((RESULT_DIR / "laufinfo.json").read_text(encoding="utf-8"))
if info.get("status") != "completed" or info.get("resolution_m") != CELL_SIZE:
    raise ValueError("Bitte einen abgeschlossenen Lauf mit passender Auflösung auswählen.")
coefficients = pd.read_csv(RESULT_DIR / "koeffizienten.csv")
tests = pd.read_csv(RESULT_DIR / "hypothesentests.csv")
diagnostics = pd.read_csv(RESULT_DIR / "diagnostik.csv")
sample = pd.read_csv(RESULT_DIR / "stichprobe.csv")
plotdata = pd.read_parquet(RESULT_DIR / "plotdaten.parquet")
covariances = pd.read_parquet(RESULT_DIR / "kovarianzen.parquet")
if coefficients.duplicated(["model", "term"]).any():
    raise ValueError("Doppelte Modellkoeffizienten im Export.")
if covariances.duplicated(["model", "covariance", "term_i", "term_j"]).any():
    raise ValueError("Doppelte Kovarianzeinträge im Export.")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
display(sample)
display(tests)
display(diagnostics.loc[diagnostics.status.isin(["FAILED", "REVIEW", "unidentified"])])
print("Ergebnisquelle:", RESULT_DIR)
print("Altersabhängige Sensitivität:", info.get("age_sensitivity", {}))
print("30-Minuten-Cox-Sensitivität:", info.get("accessibility_sensitivity", {}))


## 2. Abbildungen

Die Intervalle beziehen sich auf die jeweils ausgewiesene Kovarianzschätzung.
H4-Abfahrtsvergleiche halten die anderen Merkmale und die Haltestellenverfügbarkeit
konstant. Ein gemeinsamer Test prüft die Gleichheit beziehungsweise Nullhypothese,
keine gemeinsame Wirkungsrichtung.


In [ ]:
def save_figure(fig, name):
    fig.tight_layout()
    for extension in ["png", "svg"]:
        fig.savefig(OUTPUT_DIR / f"{name}.{extension}", dpi=600, bbox_inches="tight")
    plt.close(fig)


def forest(table, labels, filename, hazard=False):
    table = table.dropna(subset=["estimate", "ci_lower", "ci_upper"])
    if table.empty: return
    fig, ax = plt.subplots(figsize=(6.5, max(2.8, .45*len(table)+1.2)))
    point = np.exp(table.estimate) if hazard else table.estimate
    lower = np.exp(table.ci_lower) if hazard else table.ci_lower
    upper = np.exp(table.ci_upper) if hazard else table.ci_upper
    ax.errorbar(point, np.arange(len(table)), xerr=[point-lower, upper-point],
        fmt="o", color="black", capsize=3, markersize=4)
    ax.axvline(1 if hazard else 0, color=".5", linestyle="--", linewidth=.8)
    ax.set_yticks(np.arange(len(table)), [labels(row) for row in table.itertuples()])
    ax.invert_yaxis()
    ax.set_xlabel("Hazard Ratio mit 95%-Konfidenzintervall" if hazard else "Koeffizient mit 95%-Konfidenzintervall")
    apa7_axes(ax)
    save_figure(fig, filename)


def subset_tests(model, identifiers):
    return tests.loc[tests.model.eq(model) & tests.hypothesis_id.isin(identifiers)]


In [ ]:
count = subset_tests("founding_nb2", ["H1", "H2a"])
labels = {"H1": "Bevölkerung bei konstantem Firmenbestand", "H2a": "Firmenbestand bei konstanter Bevölkerung"}
forest(count, lambda r: labels[r.hypothesis_id], "01_gruendungseffekte")
survival = subset_tests("survival_cox", ["H2b", "H2c"])
labels_cox = {"H2b": "Branchengleicher Bestand", "H2c": "Branchenfremder Bestand"}
forest(survival, lambda r: labels_cox[r.hypothesis_id], "02_cox_effekte", hazard=True)

h3 = plotdata.loc[plotdata.plot_type.eq("h3_population")].copy()
if not h3.empty:
    h3["estimate"] = h3.population_effect
    forest(h3, lambda r: {"A":"Hohe lokale Nachfrageabhängigkeit", "C":"Geringe lokale Nachfrageabhängigkeit"}[r.group], "03_h3_gruppeneffekte")

h4 = plotdata.loc[plotdata.plot_type.eq("h4_frequency")].sort_values("departures")
if not h4.empty:
    fig, ax = plt.subplots(figsize=(6.5, 3.5))
    ax.plot(h4.departures, h4.relative_count, color="black")
    ax.fill_between(h4.departures.to_numpy(dtype=float), h4.ci_lower.to_numpy(dtype=float),
                    h4.ci_upper.to_numpy(dtype=float), color=".8")
    ax.axhline(1, color=".5", linestyle="--", linewidth=.8)
    ax.set_xscale("log")
    ax.set(xlabel="Erreichbare werktägliche Abfahrten (logarithmische Achse)",
           ylabel="Relative erwartete Gründungszahl\nReferenz: "+f"{h4.reference.iloc[0]:g} Abfahrten")
    apa7_axes(ax)
    save_figure(fig, "04_h4_abfahrtsfrequenz")

h5 = subset_tests("h5_sectors", ["H5_sector"])
h5 = h5.loc[h5.covariance.eq("cell")].sort_values("sparte")
forest(h5, lambda r: r.sparte_name, "05_h5_sparten")


In [ ]:
models = ["founding_nb2", "founding_stock_squared", "founding_access_30min", "founding_centre_control", "founding_without_2020"]
names = {"founding_nb2":"Hauptmodell", "founding_stock_squared":"Quadratischer Eigenbestand",
         "founding_access_30min":"30 Minuten", "founding_centre_control":"Mit Zentrumsnähe",
         "founding_without_2020":"Ohne Übergangsjahr 2020"}
comparison = tests.loc[tests.model.isin(models) & tests.hypothesis_id.eq("H1")].copy()
comparison["order"] = comparison.model.map({m:i for i,m in enumerate(models)})
forest(comparison.sort_values("order"), lambda r: names[r.model], "06_h1_sensitivitaeten")
moran = plotdata.loc[plotdata.plot_type.eq("moran")].sort_values("period")
if not moran.empty:
    fig, ax = plt.subplots(figsize=(6.5, 3.3))
    ax.plot(np.arange(len(moran)), moran.I, "o-", color="black", linewidth=1)
    ax.axhline(0, color=".5", linewidth=.8)
    ax.set_xticks(np.arange(len(moran)), moran.period, rotation=25)
    ax.set(xlabel="Ausgewählte Prüfquartale", ylabel="Moran’s I der Pearson-Residuen")
    apa7_axes(ax)
    save_figure(fig, "07_raeumliche_residuen")


## 3. Überblick und Modellgrenzen


In [ ]:
def md_table(frame):
    frame = frame.fillna("")
    lines = ["| " + " | ".join(map(str, frame.columns)) + " |",
             "| " + " | ".join(["---"]*len(frame.columns)) + " |"]
    lines += ["| " + " | ".join(str(v).replace("|", "/").replace("\n", " ") for v in row) + " |"
              for row in frame.itertuples(index=False, name=None)]
    return "\n".join(lines)

selected = tests.loc[tests.model.isin(["founding_nb2", "survival_cox", "h3_A_vs_C", "h5_sectors"])].copy()
columns = [c for c in ["model", "hypothesis_id", "sparte_name", "covariance", "estimate", "ci_lower", "ci_upper", "statistic", "df", "p_value", "p_holm_sectors", "coverage"] if c in selected]
overview = "# Ergebnisüberblick\n\n"+md_table(selected[columns].round(4))
overview += "\n\n## Stichprobe\n\n"+md_table(sample)
flags = diagnostics.loc[diagnostics.status.isin(["FAILED", "REVIEW", "unidentified"])]
overview += "\n\n## Prüfbedarf\n\n"+md_table(flags)
overview += ("\n\nAlle Zusammenhänge sind bedingt auf die Modellspezifikation. Zell- und räumliche Blockintervalle "
    "sowie PH- und Funktionsformprüfungen sind gemeinsam zu beurteilen. H4 ist gemäß der aktualisierten "
    "Definition operationalisiert; dies bedeutet keine empirische Bestätigung. H5-Spartenunterschiede "
    "werden mit dem gemeinsamen Gleichheitstest beurteilt. Nicht schätzbare Sparten bleiben ausgewiesen. "
    "Bevölkerungsrückrechnung und jährliche Infrastruktursnapshots begrenzen die zeitliche Interpretation.\n")
overview += "\nAltersabhängige Sensitivität: "+json.dumps(info.get("age_sensitivity", {}), ensure_ascii=False)
overview += "\n30-Minuten-Cox-Sensitivität: "+json.dumps(info.get("accessibility_sensitivity", {}), ensure_ascii=False)
(OUTPUT_DIR / "ergebnisuebersicht.md").write_text(overview+"\n", encoding="utf-8")
print("Abbildungen und Überblick:", OUTPUT_DIR)
